In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

In [2]:
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

AIR_QUALITY_DIR = RAW_DIR / 'openaq_hourly'
LOCAL_TZ = 'Europe/Skopje'
MIN_HOURS_PER_DAY = 18
POLLUTANT_ORDER = ['pm25', 'pm10', 'no2', 'o3', 'so2', 'co', 'no']

In [3]:
air_files = sorted(
    path for path in AIR_QUALITY_DIR.rglob('*.parquet')
    if path.name != 'download_log.parquet'
)

print(f'Air-quality files found: {len(air_files)}')

if not air_files:
    raise FileNotFoundError(f'No files found in {AIR_QUALITY_DIR}')

Air-quality files found: 271


In [4]:
def find_timestamp_column(df):
    candidates = [
        'period.datetimeFrom.utc',
        'period.datetimeFrom.local',
        'datetimeFrom.utc',
        'datetimeFrom.local',
        'time',
    ]
    return next((col for col in candidates if col in df.columns), None)

daily_parts = []
skipped_files = []

for i, file_path in enumerate(air_files, start=1):
    df = pd.read_parquet(file_path)

    if df.empty:
        skipped_files.append({'file': str(file_path), 'reason': 'empty_file'})
        continue

    timestamp_column = find_timestamp_column(df)
    required = [
        'source_location_id',
        'source_location_name',
        'source_sensor_id',
        'source_parameter',
        'value',
    ]
    missing = [col for col in required if col not in df.columns]

    if timestamp_column is None or missing:
        skipped_files.append({
            'file': str(file_path),
            'reason': f'timestamp={timestamp_column}; missing={missing}',
        })
        continue

    clean = pd.DataFrame({
        'station_id': df['source_location_id'],
        'station_name': df['source_location_name'],
        'sensor_id': df['source_sensor_id'],
        'parameter': df['source_parameter'].astype(str).str.lower().str.strip(),
        'timestamp_utc': pd.to_datetime(df[timestamp_column], utc=True, errors='coerce'),
        'value': pd.to_numeric(df['value'], errors='coerce'),
    })

    clean = clean.dropna(subset=['station_id', 'station_name', 'parameter', 'timestamp_utc', 'value'])
    clean = clean[clean['parameter'].isin(POLLUTANT_ORDER)].copy()

    if clean.empty:
        continue

    local_time = clean['timestamp_utc'].dt.tz_convert(LOCAL_TZ)
    clean['date'] = local_time.dt.tz_localize(None).dt.normalize()
    clean['hour'] = local_time.dt.tz_localize(None).dt.floor('h')

    hourly = (
        clean.groupby(['station_id', 'station_name', 'sensor_id', 'parameter', 'date', 'hour'], as_index=False)
        .agg(value=('value', 'mean'))
    )

    daily = (
        hourly.groupby(['station_id', 'station_name', 'parameter', 'date'], as_index=False)
        .agg(
            daily_mean=('value', 'mean'),
            daily_median=('value', 'median'),
            daily_min=('value', 'min'),
            daily_max=('value', 'max'),
            daily_std=('value', 'std'),
            observed_hours=('hour', 'nunique'),
        )
    )

    daily['is_valid_day'] = daily['observed_hours'] >= MIN_HOURS_PER_DAY
    daily.loc[~daily['is_valid_day'], ['daily_mean', 'daily_median', 'daily_min', 'daily_max', 'daily_std']] = np.nan
    daily_parts.append(daily)

    if i % 25 == 0 or i == len(air_files):
        print(f'Processed {i}/{len(air_files)} files')

Processed 25/271 files


Processed 50/271 files


Processed 75/271 files


Processed 100/271 files


Processed 125/271 files


...

Processed 175/271 files


Processed 200/271 files


Processed 225/271 files


Processed 250/271 files


Processed 271/271 files


In [5]:
if not daily_parts:
    raise RuntimeError('No usable daily air-quality data was created.')

air_daily_long_df = pd.concat(daily_parts, ignore_index=True)
air_daily_long_df = air_daily_long_df.sort_values(['station_id', 'parameter', 'date']).reset_index(drop=True)

print('Daily long shape:', air_daily_long_df.shape)
display(air_daily_long_df.head())

Daily long shape: (58685, 11)


,station_id,station_name,parameter,date,daily_mean,daily_median,daily_min,daily_max,daily_std,observed_hours,is_valid_day
0,2919,Gazi Baba,co,2016-12-10,NaN,NaN,NaN,NaN,NaN,4,False
1,2919,Gazi Baba,co,2016-12-11,3393.636364,3375.0,1300.0,6510.0,1404.525154,22,True
2,2919,Gazi Baba,co,2016-12-12,2133.526316,1470.0,661.0,4280.0,1248.349683,19,True
3,2919,Gazi Baba,co,2016-12-13,440.333333,441.0,371.0,499.0,34.817170,18,True
4,2919,Gazi Baba,co,2016-12-14,1395.272727,1026.5,441.0,3240.0,1004.271694,22,True


In [6]:
skipped_files_df = pd.DataFrame(skipped_files)
skipped_files_df.to_parquet(PROCESSED_DIR / 'air_quality_skipped_files.parquet', index=False)
print(f'Skipped files: {len(skipped_files_df)}')
if len(skipped_files_df) != 0:
    display(skipped_files_df.head())

Skipped files: 0


In [7]:
key_columns = ['station_id', 'station_name', 'date']

values_df = (
    air_daily_long_df.pivot_table(
        index=key_columns,
        columns='parameter',
        values='daily_mean',
        aggfunc='mean',
    )
    .reset_index()
)
values_df.columns.name = None

hours_df = (
    air_daily_long_df.pivot_table(
        index=key_columns,
        columns='parameter',
        values='observed_hours',
        aggfunc='max',
    )
    .add_suffix('_observed_hours')
    .reset_index()
)
hours_df.columns.name = None

validity_df = (
    air_daily_long_df.pivot_table(
        index=key_columns,
        columns='parameter',
        values='is_valid_day',
        aggfunc='max',
    )
    .add_suffix('_valid_day')
    .reset_index()
)
validity_df.columns.name = None

air_daily_wide_df = (
    values_df
    .merge(hours_df, on=key_columns, how='left', validate='one_to_one')
    .merge(validity_df, on=key_columns, how='left', validate='one_to_one')
    .sort_values(['station_id', 'date'])
    .reset_index(drop=True)
)

print('Daily wide shape:', air_daily_wide_df.shape)
display(air_daily_wide_df.head())

Daily wide shape: (12271, 24)


,station_id,station_name,date,co,no,no2,o3,pm10,pm25,so2,...,pm10_observed_hours,pm25_observed_hours,so2_observed_hours,co_valid_day,no_valid_day,no2_valid_day,o3_valid_day,pm10_valid_day,pm25_valid_day,so2_valid_day
0,2919,Gazi Baba,2016-12-11,3393.636364,NaN,11.022190,NaN,334.590909,NaN,5.207727,...,22.0,NaN,22.0,True,NaN,True,NaN,True,NaN,True
1,2919,Gazi Baba,2016-12-12,2133.526316,NaN,7.732222,NaN,187.505263,NaN,5.621053,...,19.0,NaN,19.0,True,NaN,True,NaN,True,NaN,True
2,2919,Gazi Baba,2016-12-13,440.333333,NaN,NaN,NaN,16.307778,NaN,3.805000,...,18.0,NaN,18.0,True,NaN,False,NaN,True,NaN,True
3,2919,Gazi Baba,2016-12-14,1395.272727,NaN,6.492524,NaN,115.007273,NaN,4.334091,...,22.0,NaN,22.0,True,NaN,True,NaN,True,NaN,True
4,2919,Gazi Baba,2016-12-15,1561.611111,NaN,NaN,NaN,146.438889,NaN,6.127222,...,18.0,NaN,18.0,True,NaN,False,NaN,True,NaN,True


In [8]:
weather_files = sorted(
    path for path in RAW_DIR.glob('weather_hourly_skopje_*.parquet')
    if re.fullmatch(r'weather_hourly_skopje_\d{4}\.parquet', path.name)
)

print(f'Weather yearly files found: {len(weather_files)}')

if not weather_files:
    raise FileNotFoundError('No yearly weather files found in data/raw.')

weather_hourly_df = pd.concat([pd.read_parquet(path) for path in weather_files], ignore_index=True)
weather_hourly_df['time'] = pd.to_datetime(weather_hourly_df['time'], errors='coerce')
weather_hourly_df = weather_hourly_df.dropna(subset=['time']).copy()
weather_hourly_df['date'] = weather_hourly_df['time'].dt.normalize()

print('Weather hourly shape:', weather_hourly_df.shape)

Weather yearly files found: 10


Weather hourly shape: (87672, 41)


In [9]:
def mode_or_nan(values):
    values = values.dropna()
    return values.mode().iloc[0] if not values.empty else np.nan

weather_daily_df = (
    weather_hourly_df.groupby('date', as_index=False)
    .agg(
        temperature_2m_mean=('temperature_2m', 'mean'),
        temperature_2m_min=('temperature_2m', 'min'),
        temperature_2m_max=('temperature_2m', 'max'),
        relative_humidity_2m_mean=('relative_humidity_2m', 'mean'),
        relative_humidity_2m_min=('relative_humidity_2m', 'min'),
        relative_humidity_2m_max=('relative_humidity_2m', 'max'),
        dew_point_2m_mean=('dew_point_2m', 'mean'),
        apparent_temperature_mean=('apparent_temperature', 'mean'),
        pressure_msl_mean=('pressure_msl', 'mean'),
        pressure_msl_min=('pressure_msl', 'min'),
        pressure_msl_max=('pressure_msl', 'max'),
        surface_pressure_mean=('surface_pressure', 'mean'),
        precipitation_sum=('precipitation', 'sum'),
        rain_sum=('rain', 'sum'),
        snowfall_sum=('snowfall', 'sum'),
        cloud_cover_mean=('cloud_cover', 'mean'),
        cloud_cover_low_mean=('cloud_cover_low', 'mean'),
        cloud_cover_mid_mean=('cloud_cover_mid', 'mean'),
        cloud_cover_high_mean=('cloud_cover_high', 'mean'),
        shortwave_radiation_sum=('shortwave_radiation', 'sum'),
        direct_radiation_sum=('direct_radiation', 'sum'),
        diffuse_radiation_sum=('diffuse_radiation', 'sum'),
        global_tilted_irradiance_sum=('global_tilted_irradiance', 'sum'),
        sunshine_duration_sum=('sunshine_duration', 'sum'),
        wind_speed_10m_mean=('wind_speed_10m', 'mean'),
        wind_speed_10m_max=('wind_speed_10m', 'max'),
        wind_speed_100m_mean=('wind_speed_100m', 'mean'),
        wind_speed_100m_max=('wind_speed_100m', 'max'),
        wind_gusts_10m_max=('wind_gusts_10m', 'max'),
        wind_direction_10m_u=('wind_direction_10m', lambda s: np.sin(np.deg2rad(s)).mean()),
        wind_direction_10m_v=('wind_direction_10m', lambda s: np.cos(np.deg2rad(s)).mean()),
        wind_direction_100m_u=('wind_direction_100m', lambda s: np.sin(np.deg2rad(s)).mean()),
        wind_direction_100m_v=('wind_direction_100m', lambda s: np.cos(np.deg2rad(s)).mean()),
        et0_fao_evapotranspiration_sum=('et0_fao_evapotranspiration', 'sum'),
        weather_code_mode=('weather_code', mode_or_nan),
        snow_depth_mean=('snow_depth', 'mean'),
        vapour_pressure_deficit_mean=('vapour_pressure_deficit', 'mean'),
        soil_temperature_0_to_7cm_mean=('soil_temperature_0_to_7cm', 'mean'),
        soil_temperature_7_to_28cm_mean=('soil_temperature_7_to_28cm', 'mean'),
        soil_temperature_28_to_100cm_mean=('soil_temperature_28_to_100cm', 'mean'),
        soil_temperature_100_to_255cm_mean=('soil_temperature_100_to_255cm', 'mean'),
        soil_moisture_0_to_7cm_mean=('soil_moisture_0_to_7cm', 'mean'),
        soil_moisture_7_to_28cm_mean=('soil_moisture_7_to_28cm', 'mean'),
        soil_moisture_28_to_100cm_mean=('soil_moisture_28_to_100cm', 'mean'),
        soil_moisture_100_to_255cm_mean=('soil_moisture_100_to_255cm', 'mean'),
    )
)

print('Weather daily shape:', weather_daily_df.shape)
display(weather_daily_df.head())

Weather daily shape: (3653, 46)


,date,temperature_2m_mean,temperature_2m_min,temperature_2m_max,relative_humidity_2m_mean,relative_humidity_2m_min,relative_humidity_2m_max,dew_point_2m_mean,apparent_temperature_mean,pressure_msl_mean,...,snow_depth_mean,vapour_pressure_deficit_mean,soil_temperature_0_to_7cm_mean,soil_temperature_7_to_28cm_mean,soil_temperature_28_to_100cm_mean,soil_temperature_100_to_255cm_mean,soil_moisture_0_to_7cm_mean,soil_moisture_7_to_28cm_mean,soil_moisture_28_to_100cm_mean,soil_moisture_100_to_255cm_mean
0,2016-01-01,-1.587500,-5.5,2.5,36.458333,26,46,-14.775000,-5.558333,1029.204167,...,0.000000,0.356250,-0.508333,1.337500,4.775000,11.483333,0.316000,0.321000,0.32,0.29475
1,2016-01-02,1.529167,-1.1,5.3,48.250000,39,59,-8.341667,-2.208333,1024.137500,...,0.000000,0.357500,-0.212500,1.133333,4.595833,11.350000,0.316000,0.320958,0.32,0.29500
2,2016-01-03,0.591667,-0.7,2.2,80.166667,60,98,-2.541667,-2.716667,1013.625000,...,0.015000,0.127083,0.187500,1.100000,4.437500,11.220833,0.316292,0.320000,0.32,0.29500
3,2016-01-04,0.645833,-1.8,3.5,84.750000,64,98,-1.700000,-2.425000,1006.133333,...,0.047500,0.097917,0.516667,1.116667,4.283333,11.070833,0.323292,0.320000,0.32,0.29500
4,2016-01-05,4.979167,1.1,8.2,87.000000,75,97,2.950000,2.820833,1003.716667,...,0.065833,0.123333,0.925000,1.220833,4.170833,10.941667,0.362375,0.323833,0.32,0.29500


In [10]:
final_daily_df = (
    air_daily_wide_df
    .merge(
        weather_daily_df,
        on='date',
        how='left',
        validate='many_to_one',
        indicator='weather_merge_status',
    )
    .sort_values(['station_id', 'date'])
    .reset_index(drop=True)
)

print(final_daily_df['weather_merge_status'].value_counts())
final_daily_df = final_daily_df.drop(columns='weather_merge_status')

final_daily_df['month'] = final_daily_df['date'].dt.month
final_daily_df['day_of_week'] = final_daily_df['date'].dt.dayofweek
final_daily_df['is_weekend'] = (final_daily_df['day_of_week'] >= 5).astype(int)
final_daily_df['season'] = final_daily_df['month'].map({
    12: 'winter', 1: 'winter', 2: 'winter',
    3: 'spring', 4: 'spring', 5: 'spring',
    6: 'summer', 7: 'summer', 8: 'summer',
    9: 'autumn', 10: 'autumn', 11: 'autumn',
})
final_daily_df['rain_indicator'] = (final_daily_df['precipitation_sum'] > 0).astype(int)
final_daily_df['wind_category'] = pd.cut(
    final_daily_df['wind_speed_10m_mean'],
    bins=[-np.inf, 2, 5, np.inf],
    labels=['weak', 'moderate', 'strong'],
)

print('Final daily shape:', final_daily_df.shape)
display(final_daily_df.head())

weather_merge_status
both          12271
left_only         0
right_only        0
Name: count, dtype: int64
Final daily shape: (12271, 75)


,station_id,station_name,date,co,no,no2,o3,pm10,pm25,so2,...,soil_moisture_0_to_7cm_mean,soil_moisture_7_to_28cm_mean,soil_moisture_28_to_100cm_mean,soil_moisture_100_to_255cm_mean,month,day_of_week,is_weekend,season,rain_indicator,wind_category
0,2919,Gazi Baba,2016-12-11,3393.636364,NaN,11.022190,NaN,334.590909,NaN,5.207727,...,0.330208,0.309000,0.229000,0.290833,12,6,1,winter,0,weak
1,2919,Gazi Baba,2016-12-12,2133.526316,NaN,7.732222,NaN,187.505263,NaN,5.621053,...,0.327667,0.309000,0.229917,0.290000,12,0,0,winter,0,moderate
2,2919,Gazi Baba,2016-12-13,440.333333,NaN,NaN,NaN,16.307778,NaN,3.805000,...,0.322375,0.308583,0.230292,0.290000,12,1,0,winter,0,moderate
3,2919,Gazi Baba,2016-12-14,1395.272727,NaN,6.492524,NaN,115.007273,NaN,4.334091,...,0.316958,0.307417,0.231000,0.290000,12,2,0,winter,0,weak
4,2919,Gazi Baba,2016-12-15,1561.611111,NaN,NaN,NaN,146.438889,NaN,6.127222,...,0.315250,0.306542,0.231792,0.290000,12,3,0,winter,0,weak


In [11]:
output_parquet = PROCESSED_DIR / "air_pollution_weather_daily_master.parquet"

final_daily_df.to_parquet(output_parquet, index=False)


generated_columns = [
    "station_id",
    "station_name",
    "date",
    "month",
    "day_of_week",
    "is_weekend",
    "season",
    "rain_indicator",
    "wind_category",
]

generated_columns += [
    column
    for column in final_daily_df.columns
    if column.endswith("_observed_hours")
    or column.endswith("_valid_day")
]

downloaded_columns = [
    column
    for column in final_daily_df.columns
    if column not in generated_columns
]


missingness_df = pd.DataFrame({
    "missing_count": final_daily_df[downloaded_columns].isna().sum(),
    "missing_percent": (
        final_daily_df[downloaded_columns].isna().mean() * 100
    ).round(2),
    "data_type": final_daily_df[downloaded_columns].dtypes.astype(str),
}).sort_values("missing_percent", ascending=False)


missingness_df.to_parquet(
    PROCESSED_DIR / "downloaded_columns_missingness.parquet"
)


print(f"Saved full dataset: {output_parquet}")
print(f"Rows: {len(final_daily_df):,}")
print(f"All columns: {len(final_daily_df.columns):,}")
print(f"Downloaded-data columns shown: {len(downloaded_columns):,}")

display(missingness_df)

Saved full dataset: ../data/processed/air_pollution_weather_daily_master.parquet
Rows: 12,271
All columns: 75
Downloaded-data columns shown: 52


,missing_count,missing_percent,data_type
no,11367,92.63,float64
pm25,8276,67.44,float64
so2,5166,42.10,float64
co,4766,38.84,float64
o3,4362,35.55,float64
pm10,3380,27.54,float64
no2,3045,24.81,float64
temperature_2m_mean,0,0.00,float64
temperature_2m_min,0,0.00,float64
temperature_2m_max,0,0.00,float64
